# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

We enumerate all record sets defined in the schema, along with their corresponding fields and columns. All references use explicit `@id` identifiers.

In [ ]:
# List all available record sets with their @ids
record_sets = list(dataset.record_set_ids())  # Returns a list of all RecordSet @ids
print("Available Record Sets (@id):")
for i, rs_id in enumerate(record_sets):
    print(f"  {i+1}: {rs_id}")
    rs = dataset.get_record_set(rs_id)
    field_ids = rs.field_ids()
    print("    Fields (@id):")
    for f_id in field_ids:
        field = dataset.get_field(f_id)
        print(f"      - {f_id} (name: {field.name}, type: {field.data_type})")
    column_ids = rs.column_ids() if hasattr(rs, 'column_ids') else []
    if column_ids:
        print("    Columns (@id):")
        for c_id in column_ids:
            print(f"      - {c_id}")


## Example: Inspect Data Records
Let's print some records from a specific record set for inspection. Adjust the record set `@id` as needed. By default, we select the first listed record set.

In [ ]:
# Print a handful of records from the first record set
if not record_sets:
    raise Exception("No record sets found in the dataset.")
example_record_set = record_sets[0]
print(f"\nShowing 3 records from RecordSet: {example_record_set}")
for i, rec in enumerate(dataset.records(record_set=example_record_set)):
    pprint.pprint(rec)
    if i>=2:
        break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set in record_sets:
    print(f"Loading records from RecordSet: {record_set}")
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    if not df.empty:
        dataframes[record_set] = df
        print(f"  --> Loaded {len(df)} rows and {len(df.columns)} columns.")
    else:
        print("  --> No records found.")

# Show the columns (field @id) for the main record set
main_record_set = example_record_set
if main_record_set in dataframes:
    print(f"\nFields (@id) as DataFrame columns for RecordSet {main_record_set}:")
    print(dataframes[main_record_set].columns.tolist())
    display(dataframes[main_record_set].head())
else:
    print(f"No records loaded for record set: {main_record_set}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping/categorizing data. All fields are referenced via their `@id`.

In [ ]:
# Identify a numeric field by @id, fallback to guessing if field metadata is present
rs = dataset.get_record_set(main_record_set)
numeric_field_id = None
for f_id in rs.field_ids():
    field = dataset.get_field(f_id)
    if getattr(field, 'data_type', None) in ['Number', 'Float', 'Integer', 'schema:Number', 'schema:Float', 'schema:Integer']:
        numeric_field_id = f_id
        break
if not numeric_field_id:
    # Fallback: guess by column name
    for col in dataframes[main_record_set].columns:
        if 'abundance' in col.lower() or 'intensity' in col.lower() or 'count' in col.lower() or 'mw' in col.lower():
            numeric_field_id = col
            break

if not numeric_field_id:
    raise Exception("No numeric field could be automatically detected. Please specify by @id.")
print(f"Numeric field selected for EDA: {numeric_field_id}")
numeric_field = numeric_field_id

# Remove rows with missing/NaN in the chosen numeric field
df = dataframes[main_record_set]
df_clean = df[df[numeric_field].notna()].copy()

# Set threshold as the 75th percentile
threshold = df_clean[numeric_field].astype(float).quantile(0.75)
filtered_df = df_clean[df_clean[numeric_field].astype(float) > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
    ) / filtered_df[numeric_field].astype(float).std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field (guess from non-numeric fields)
group_field = None
for f_id in rs.field_ids():
    field = dataset.get_field(f_id)
    if getattr(field, 'data_type', None) in ['Text', 'String', 'schema:Text', 'schema:String']:
        group_field = f_id
        break
if group_field and group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nMean {numeric_field} grouped by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of numeric field
plt.figure(figsize=(7,4))
sns.histplot(df_clean[numeric_field].astype(float), bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

if group_field and group_field in df_clean.columns:
    top_groups = df_clean[group_field].value_counts().nlargest(5).index
    plt.figure(figsize=(8,5))
    sns.boxplot(data=df_clean[df_clean[group_field].isin(top_groups)], x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field} (top 5 groups)")
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and process a complex scientific dataset using the `mlcroissant` library.

- We inspected available record sets, fields, and their unique `@id` references.
- Records were extracted to DataFrames, explored, filtered, normalized, and visualized.
- All access used strict use of schema `@id` for fields, ensuring reproducible and robust data handling.

The `mlcroissant` library allows programmatic, transparent, FAIR-aligned access to rich scientific data with schema awareness. Explore further by adjusting filtering and analysis using other field `@id`s!